# Feature Engineering on Primary Land Use Tax Lot Output (PLUTO) and Driver Revenue Datasets:

----

# Import Libraries:

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import geopandas as gpd
from shapely import wkt
import pandas as pd 
import os

In [2]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("feature_engineering_pluto+revenue")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

24/08/21 01:30:53 WARN Utils: Your hostname, Cocos-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 172.16.33.67 instead (on interface en0)
24/08/21 01:30:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 01:30:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/21 01:30:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# Read Files:

In [3]:
base_dir = "../data"

PLUTO dataset:

In [4]:
pluto_df_path = base_dir + '/developed/merged_data/pluto_df.csv'
pluto_df = pd.read_csv(pluto_df_path)
pluto_df.head()

,building_class,geometry,location_id,service_zone,zone,borough
0,F,POINT (-74.0750604 40.6288711),221,Boro Zone,Stapleton,Staten Island
1,B,POINT (-74.0746779 40.6154356),221,Boro Zone,Stapleton,Staten Island
2,B,POINT (-74.074487 40.6155098),221,Boro Zone,Stapleton,Staten Island
3,B,POINT (-74.0745662 40.6154741),221,Boro Zone,Stapleton,Staten Island
4,B,POINT (-74.0750307 40.6152734),221,Boro Zone,Stapleton,Staten Island


Zone dataset:

In [5]:
zone_gdf_path = base_dir + '/developed/merged_data/zone_gdf.csv'
zone_gdf = pd.read_csv(zone_gdf_path)
zone_gdf['geometry'] = zone_gdf['geometry'].apply(wkt.loads)
zone_gdf.head()

,location_id,service_zone,Shape_Leng,Shape_Area,zone,borough,geometry
0,1,EWR,0.116357,0.000782,Newark Airport,EWR,POLYGON ((-74.18445299999996 40.69499600000009...
1,2,Boro Zone,0.433470,0.004866,Jamaica Bay,Queens,MULTIPOLYGON (((-73.82337597260663 40.63898704...
2,3,Boro Zone,0.084341,0.000314,Allerton/Pelham Gardens,Bronx,"POLYGON ((-73.84792614099985 40.8713422340001,..."
3,4,Yellow Zone,0.043567,0.000112,Alphabet City,Manhattan,POLYGON ((-73.97177410965318 40.72582128133726...
4,5,Boro Zone,0.092146,0.000498,Arden Heights,Staten Island,POLYGON ((-74.17421738099989 40.56256808600009...


Daily revenue:

In [6]:
daily_revenue_sdf_path = base_dir + '/developed/merged_data/daily_revenue'
daily_revenue_sdf = spark.read.parquet(daily_revenue_sdf_path)
daily_revenue_df = daily_revenue_sdf.toPandas()
daily_revenue_df.head()

,pickup_date,PULocationID,daily_revenue
0,2023-07-01,2,0.243696
1,2023-07-01,3,102.384565
2,2023-07-01,4,230.421413
3,2023-07-01,5,19.353261
4,2023-07-01,6,34.761359


# Find Daily Revenue by Building Class:

In [7]:
# Merge `pluto_df` and `daily_revenue_df` on `location_id` and `PULocationID`
daily_revenue_by_building_class_df = pd.merge(pluto_df, daily_revenue_df, 
                                              left_on='location_id', 
                                              right_on='PULocationID', 
                                              how='left')

# Group by `building_class`, and calculate the sum of `daily_revenued`
daily_revenue_by_building_class_df = daily_revenue_by_building_class_df.groupby(['building_class'])['daily_revenue'] \
                                                                       .sum() \
                                                                       .reset_index() 

# Sort by daily revenue
daily_revenue_by_building_class_df = daily_revenue_by_building_class_df.sort_values(by='daily_revenue', ascending=False)\
                                                                       .reset_index(drop=True)

daily_revenue_by_building_class_df.head()

,building_class,daily_revenue
0,B,9.394588e+09
1,A,7.844196e+09
2,C,7.462351e+09
3,S,1.823149e+09
4,D,9.574618e+08


In [8]:
daily_revenue_by_building_class_df.head(24)

,building_class,daily_revenue
0,B,9.394588e+09
1,A,7.844196e+09
2,C,7.462351e+09
3,S,1.823149e+09
4,D,9.574618e+08
5,K,9.084983e+08
6,V,7.443072e+08
7,R,7.005228e+08
8,G,5.727261e+08
9,O,4.516237e+08


# Find Average Daily Revenue by Location ID:

In [9]:
zone_gdf = gpd.GeoDataFrame(zone_gdf, geometry='geometry')
zone_gdf = zone_gdf.drop_duplicates('location_id')

In [10]:
# Merge `zone_gdf` and `daily_revenue_df` on `location_id` and `PULocationID`
daily_revenue_by_location_df = pd.merge(zone_gdf, daily_revenue_df, 
                                        left_on='location_id', 
                                        right_on='PULocationID', 
                                        how='left')

# Drop the 'PULocationID' column as it is no longer needed
daily_revenue_by_location_df = daily_revenue_by_location_df.drop('PULocationID', axis=1)

# Group by `location_id` and calculate the sum of `daily_revenue`
daily_revenue_by_location_df = daily_revenue_by_location_df.groupby(['location_id'])['daily_revenue'] \
                                                           .sum() \
                                                           .reset_index()

# Sort the DataFrame by `daily_revenue` in descending order
daily_revenue_by_location_df = daily_revenue_by_location_df.sort_values(by='daily_revenue', ascending=False) \
                                                           .reset_index(drop=True)

daily_revenue_by_location_df.head()

,location_id,daily_revenue
0,132,490265.603750
1,138,400708.996413
2,230,182360.321467
3,161,163863.122609
4,79,156364.447935


In [11]:
daily_revenue_by_location_df.head(24)

,location_id,daily_revenue
0,132,490265.603750
1,138,400708.996413
2,230,182360.321467
3,161,163863.122609
4,79,156364.447935
5,68,154724.583424
6,231,148201.997989
7,164,143869.588261
8,246,138824.617663
9,61,135972.474674


# Save the Merged Dataset:

Daily revenue by different building classes:

In [12]:
revenue_by_building_class_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue_by_building_class_df.csv'
revenue_by_building_class_df_path = os.path.join(revenue_by_building_class_df_dir, file_name)
daily_revenue_by_building_class_df.to_csv(revenue_by_building_class_df_path, index=False)

Daily revenue by different location ID:

In [13]:
revenue_by_location_df_dir = base_dir + '/developed/merged_data'
file_name = 'daily_revenue_by_location_df.csv'
revenue_by_location_df_path = os.path.join(revenue_by_location_df_dir, file_name)
daily_revenue_by_location_df.to_csv(revenue_by_location_df_path, index=False)